In [1]:
import neptune
import pandas as pd
import plotly.express as px

API_TOKEN="eyJhcGlfYWRkcmVzcyI6Imh0dHBzOi8vYXBwLm5lcHR1bmUuYWkiLCJhcGlfdXJsIjoiaHR0cHM6Ly9hcHAubmVwdHVuZS5haSIsImFwaV9rZXkiOiJhZDg5ZTI2ZS00MWUyLTRkMTUtYTEzMC01OTVhYzE1ZWVmYzIifQ=="
PROJECT="pmtest/llm-random"

In [2]:
def get_log(exp_id:int, metric:str, project=PROJECT, api_token=API_TOKEN) -> pd.DataFrame:
    run = neptune.init_run(
        project=project,
        with_id=f"LLMRANDOM-{str(exp_id)}",
        api_token=api_token,
    )
    loss_df = run[metric].fetch_values()
    run.stop()
    return loss_df

def interval_mean(df:pd.DataFrame, key="value", lag=100, sive_every_lag=True, last_record=True):
    # df[f'{key}_{lag}'] = df[key].rolling(window=lag).mean()
    df = df.copy()
    df[key] = df[key].rolling(window=lag).mean()
    if sive_every_lag: 
        every_lag_step = df[(df['step'] % lag == 0) & (df['step']!=0)]
        if last_record:
            every_lag_step = pd.concat([every_lag_step, df.iloc[[-1]]])
    else:
        every_lag_step = df

    return every_lag_step

In [ ]:


# NEMO COMPs IDs
nemo_pr_832_t = 35393
nemo_pr_768_t = 35392
nemo_pr_256_t = 35394

nemo_def_1024_long = 35461
nemo_def_1024_long_learned_emb = 35460
nemo_def_1024 = 35462
nemo_def_1024_learned_emb = 35459

nemo_def_1024_better_data = 35607
nemo_def_1024_better_data_learned_emb = 35669
nemo_def_1024_better_data_learned_emb_nhln = 35695

nemo_def_1024_relu = 35772
nemo_def_1024_relu_nhln = 35771
nemo_def_1024_nhln = 35770
nemo_def_1024 = 35669

nemo_pr_832 = 123
nemo_pr_768 = 123
nemo_pr_256 = 123


# LLMR COMPs IDs
llmr_def_1024_rope =  35763
llmr_def_1024_rope_hln = 35765


# OLD
# LLMR COMPs IDs
llmr_def_1024_long = 33672
llmr_pr_832 = 34208
llmr_pr_768 = 33952

# Additional additional
llmr_prdis_768 = 34099
llmr_prdis_832 = 34379
llmr_def_1024 = 32714
llmr_def_1024_2 = 33883

llmr_def_768 = 33602
llmr_dgptp_1024_150 = 34204
llmr_dis_768 = 34000

id_name = {
    nemo_pr_832_t: "nemo_pr_832_t",
    nemo_pr_768_t: "nemo_pr_768_t",
    nemo_pr_256_t: "nemo_pr_256_t",
    llmr_def_1024_long: "llmr_def_1024_long",
    llmr_pr_832: "llmr_pr_832",
    llmr_pr_768: "llmr_pr_768",
    llmr_prdis_768: "llmr_prdis_768",
    llmr_prdis_832: "llmr_prdis_832",
    llmr_def_1024: "llmr_def_1024",
    llmr_def_1024_2: "llmr_def_1024_2",
    llmr_def_768: "llmr_def_768",
    llmr_dgptp_1024_150: "llmr_dgptp_1024_150",
    llmr_dis_768: "llmr_dis_768",
    nemo_def_1024_long: "nemo_def_1024_long",
    nemo_def_1024_long_learned_emb: "nemo_def_1024_long_learned_emb",
    nemo_def_1024: "nemo_def_1024",
    nemo_def_1024_learned_emb: "nemo_def_1024_learned_emb",
    nemo_def_1024_better_data: "nemo_def_1024_better_data",
    nemo_def_1024_better_data_learned_emb:"nemo_def_1024_better_data_learned_emb",
    nemo_def_1024_better_data_learned_emb_nhln:"nemo_def_1024_better_data_learned_emb_nhln",
    llmr_def_1024_rope:"llmr_def_1024_rope",
    llmr_def_1024_rope_hln:"llmr_def_1024_rope_hln",
    nemo_def_1024_relu:"nemo_def_1024_relu",
    nemo_def_1024_relu_nhln:"nemo_def_1024_relu_nhln",
    nemo_def_1024_nhln:"nemo_def_1024_nhln",
    nemo_def_1024:"nemo_def_1024",

}

In [4]:
NEMO_METRIC = "training/reduced_train_loss"
LLMR_METRIC = "loss_interval/1"
# NEMO_METRIC = "training/lr"
# LLMR_METRIC = "lr"

# nemo_fetch = [nemo_def_1024_long, nemo_def_1024_long_learned_emb, nemo_def_1024, nemo_def_1024_learned_emb, nemo_def_1024_long_t, nemo_def_1024_long_learned_emb_t, nemo_def_1024_t, nemo_pr_832_t, nemo_pr_768_t, nemo_pr_256_t]
# llmr_fetch = [llmr_def_1024_long, llmr_pr_832, llmr_pr_768, llmr_prdis_768, llmr_prdis_832, llmr_def_1024, llmr_def_1024_2, llmr_def_768, llmr_dgptp_1024_150, llmr_dis_768]
nemo_fetch = [nemo_def_1024_relu, nemo_def_1024_relu_nhln, nemo_def_1024_nhln, nemo_def_1024]
llmr_fetch = [llmr_def_1024_long, llmr_def_1024, llmr_def_1024_2, llmr_def_1024_rope, llmr_def_1024_rope_hln]
    
comp_runs_raw = {}

def fetch_raw(exps, metric):
    res = {}
    for e in exps:
        res[e] = get_log(e, metric)
    return res

# comp_runs_raw.update(fetch_raw(llmr_fetch, LLMR_METRIC))
# comp_runs_raw.update(fetch_raw(nemo_fetch, NEMO_METRIC))
comp_llmr = fetch_raw(llmr_fetch, LLMR_METRIC)
comp_nemo = fetch_raw(nemo_fetch, NEMO_METRIC)
comp_runs_raw.update(comp_llmr)
comp_runs_raw.update(comp_nemo)

[neptune] [warning] NeptuneWarning: By default, these monitoring options are disabled in interactive sessions: 'capture_stdout', 'capture_stderr', 'capture_traceback', 'capture_hardware_metrics'. You can set them to 'True' when initializing the run and the monitoring will continue until you call run.stop() or the kernel stops. NOTE: To track the source files, pass their paths to the 'source_code' argument. For help, see: https://docs-legacy.neptune.ai/logging/source_code/


[neptune] [info   ] Neptune initialized. Open in the app: https://app.neptune.ai/pmtest/llm-random/e/LLMRANDOM-33672


Fetching loss_interval/1 values: 0 [00:00, ?/s]

[neptune] [info   ] Shutting down background jobs, please wait a moment...
[neptune] [info   ] Done!
[neptune] [info   ] All 0 operations synced, thanks for waiting!
[neptune] [info   ] Explore the metadata in the Neptune app: https://app.neptune.ai/pmtest/llm-random/e/LLMRANDOM-33672/metadata
[neptune] [info   ] Neptune initialized. Open in the app: https://app.neptune.ai/pmtest/llm-random/e/LLMRANDOM-32714


Fetching loss_interval/1 values: 0 [00:00, ?/s]

...

Fetching training/reduced_train_loss values: 0 [00:00, ?/s]

[neptune] [info   ] Shutting down background jobs, please wait a moment...
[neptune] [info   ] Done!
[neptune] [info   ] All 0 operations synced, thanks for waiting!
[neptune] [info   ] Explore the metadata in the Neptune app: https://app.neptune.ai/pmtest/llm-random/e/LLMRANDOM-35771/metadata
[neptune] [info   ] Neptune initialized. Open in the app: https://app.neptune.ai/pmtest/llm-random/e/LLMRANDOM-35770


Fetching training/reduced_train_loss values: 0 [00:00, ?/s]

[neptune] [info   ] Shutting down background jobs, please wait a moment...
[neptune] [info   ] Done!
[neptune] [info   ] All 0 operations synced, thanks for waiting!
[neptune] [info   ] Explore the metadata in the Neptune app: https://app.neptune.ai/pmtest/llm-random/e/LLMRANDOM-35770/metadata
[neptune] [info   ] Neptune initialized. Open in the app: https://app.neptune.ai/pmtest/llm-random/e/LLMRANDOM-35769


MissingFieldException: 
[95m
----MissingFieldException-------------------------------------------------------
[0m
The field "training/reduced_train_loss" was not found.

There are two possible reasons:
    - There is a typo in the path. Double-check your code for typos.
    - You are fetching a field that another process created, but the local representation is not synchronized.
    If you are sending metadata from multiple processes at the same time, synchronize the local representation before fetching values:
        [96mrun.sync()[0m

[92mNeed help?[0m-> https://docs-legacy.neptune.ai/getting_help


In [ ]:
comp_runs = comp_runs_raw.copy()

In [ ]:
for k, v in comp_runs.items():
    comp_runs[k] = interval_mean(v, lag=100)

In [ ]:
import pandas as pd
import plotly.express as px

def plot_lines(runs_ids, title:str, runs_dict = comp_runs, id_name=id_name): #, save_path:str=None
    strip_exps = []
    for e in runs_ids:
        # Create small DataFrames manually with id
        df1 = runs_dict[e].copy()
        df1['id'] = id_name[e]
        strip_exps.append(df1)
    assert len(strip_exps) > 0
    df_concat = pd.concat(strip_exps)

    # Now plot
    fig = px.line(
        df_concat,
        x='step',
        y='value',
        color='id',  # Now 'id' exists
        title=title,
        labels={'value': 'Loss', 'step': 'Training Step'}
    )

    # if save_path:
    #     fig.write_image(f"{save_path}.svg", format='svg')
    # fig.show()
    return fig

In [ ]:
# Nemo test
# fig = plot_lines([nemo_def_1024_long_learned_emb, nemo_def_1024_long_learned_emb_t], "Nemo - test")
# fig = plot_lines([nemo_def_1024_long_learned_emb, llmr_def_1024_long], "Nemo - test")
# fig = plot_lines([nemo_def_1024_better_data, nemo_def_1024_better_data_learned_emb], "Nemo - test")
# fig = plot_lines([nemo_def_1024_better_data_learned_emb, nemo_def_1024_better_data_learned_emb_nhln], "Nemo - test")
# fig = plot_lines([nemo_def_1024_better_data, llmr_def_1024_rope], "Nemo - test")
# fig = plot_lines([nemo_def_1024_better_data, llmr_def_1024_rope_hln, llmr_def_1024_rope], "Nemo - test")
fig = plot_lines([nemo_def_1024_relu, nemo_def_1024_relu_nhln, nemo_def_1024_nhln, nemo_def_1024], "Nemo - test")

fig.update_yaxes(range=[3.1, 4])
fig

In [ ]:
# Nemo test
fig = plot_lines([nemo_def_1024, nemo_def_1024_t], "Nemo - test")
# fig = plot_lines([ nemo_def_1024_learned_emb, llmr_def_1024, llmr_def_1024_2], "Nemo - test")
fig.update_yaxes(range=[3.1, 4])
fig

In [ ]:
# Nemo test
# fig = plot_lines([nemo_def_1024_long, nemo_def_1024_long_t, llmr_def_1024_long], "Nemo - test")
fig = plot_lines([nemo_def_1024, nemo_def_1024_learned_emb], "Nemo - test")
fig.write_image(f"{"charts/nemo_test"}.svg", format='svg')
fig.update_yaxes(range=[3.1, 4.5])
fig

In [ ]:
# Nemo embeddings vs 
fig = plot_lines([nemo_def_1024_long_t, nemo_def_1024_long_learned_emb_t], "Nemo - RoPE vs Learned POS")
fig.update_yaxes(range=[3.1, 4])
fig

In [ ]:
# Nemo vs LLMR - Pretraining
fig = plot_lines([llmr_def_1024_long, nemo_def_1024_long_learned_emb_t, nemo_def_1024_long_learned_emb], "Nemo vs LLMR default pretraining") # nemo_def_1024_long_learned_emb
fig.update_yaxes(range=[3.1, 4])
fig

In [ ]:
# LLMR vs LLMR - Pretraining
fig = plot_lines([llmr_def_1024, nemo_def_1024_t, llmr_def_1024_2], "Nemo vs LLMR default pretraining")
fig.update_yaxes(range=[3.2, 4])
fig.write_image(f"{"charts/frameworks_comp"}.svg", format='svg')
fig

In [ ]:
# Nemo vs LLMR - Doners
fig = plot_lines([llmr_def_1024_long, nemo_def_1024_long_t], "Nemo vs LLMR Doners")
fig.update_yaxes(range=[3.1, 4])
fig.write_image(f"{"charts/doners_comp"}.svg", format='svg')
fig

In [ ]:
# Minitron prunning vs Projected Compression in context
fig = plot_lines([llmr_pr_832, nemo_pr_832_t, llmr_def_768, llmr_def_1024, llmr_def_1024_long], "Provected Compression vs Minitron Prunning + Context: 200M -> 160M")
fig.update_yaxes(range=[3.0, 4])
fig.write_image(f"{"charts/832_comp_ctx"}.svg", format='svg')
fig

In [ ]:
# Minitron prunning vs Projected Compression
fig = plot_lines([llmr_pr_832, nemo_pr_832_t], "Provected Compression vs Minitron Prunning: 200M -> 160M")
fig.update_yaxes(range=[3.2, 4])
fig.write_image(f"{"charts/832_comp"}.svg", format='svg')
fig

In [ ]:
# Minitron prunning vs Projected Compression in context
fig = plot_lines([llmr_pr_768, nemo_pr_768_t, llmr_def_768, llmr_def_1024], "Provected Compression vs Minitron Prunning + Context: 200M -> 130M")
fig.update_yaxes(range=[3.2, 4])
fig

In [ ]:
# Minitron prunning vs Projected Compression in context
fig = plot_lines([llmr_pr_832, nemo_pr_832_t, llmr_def_768, llmr_dis_768, llmr_dgptp_1024_150], "Provected Compression vs Minitron Prunning + Context: 200M -> 160M") # , llmr_prdis_832, llmr_prdis_768
fig.update_yaxes(range=[3.2, 4])
fig.write_image(f"{"charts/all_in"}.svg", format='svg')
fig